# 02 — Research: how to improve **f1_human** and **f1_fusion**

The 01 baseline (`LogReg C=4 | word+char TF-IDF`) reaches macro-F1 ≈ 0.86, but almost all of the error is
**human ↔ fusion** confusion (f1_ai ≈ 0.99, f1_human ≈ 0.79, f1_fusion ≈ 0.80). A *fusion* text is a human text
with parts rewritten by an LLM, so it carries both signals. This notebook:

| § | What | Why |
|---|---|---|
| 1 | Error inventory of the baseline (domain, language, length, calibration) | where the errors are |
| 2 | **Print the exact misclassified texts** and the n-grams that pushed them the wrong way; diff against the human "twin" when one exists | read what the model gets wrong |
| 3 | Outside research (papers on human vs LLM markers, machine-polished and hybrid text) | ideas to test, not assumptions |
| 4 | Hand-crafted "human-ness" features: **misspellings** (and whether typos occur only in human texts), informal typography, burstiness, LLM stock words, and **segment-level inconsistency** (does part of the text look AI and part human?) | test each idea by per-domain AUC |
| 5 | Models that use the findings: TF-IDF + features, "text shape" n-grams, a stacker, a dedicated human-vs-fusion second stage, bias tuning on out-of-fold predictions | what actually moves f1_human / f1_fusion |
| 6 | Summary and recommendations | what to carry into the main pipeline |

Rules: patterns are searched for on the **training split** only, validation is used only to score, and everything is
logged to `outputs/results.csv` as experiment `02_hf_research`.

*Reference article:* L. Gaur, *"Text classification with language models: a practical guide"* (Medium). Automated
fetching of the page was blocked (HTTP 403), so it is cited for its general recipe only (pretrained encoder:
fine-tune, or embeddings + a light classifier). That is the natural next step after this notebook, but it needs
torch and ideally a GPU, which this environment does not have. Everything here is CPU-only and classical.

In [ ]:
import sys
from pathlib import Path

# make src/ importable whether the kernel starts in src/experiments, src or the repo root
SRC = next(q for p in [Path.cwd(), *Path.cwd().parents] for q in (p, p / "src") if (q / "config.py").exists())
sys.path.insert(0, str(SRC))

In [ ]:
import difflib
import gc
import re
import time
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.sparse as sp
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from spellchecker import SpellChecker

from config import LABELS, OUTPUT_DIR, SEED, SPLITS_DIR
from utils import (hstack_csr, load_rows, load_splits, load_test, log_result, macro_f1,
                   save_submission, save_val_scores)

pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 80)

EXP = "02_hf_research"
MAX_CHARS = 4_000       # what the TF-IDF models see (same as 01)
FEAT_CHARS = 8_000      # hand-crafted and segment features look a bit further into long texts
N_FOLDS = 5
TOKEN_PATTERN = r"(?u)\b\w+\b|[^\w\s]"
CLS = np.array(["ai", "fusion", "human"])   # sklearn's (alphabetical) class order for probability columns
iA, iF, iH = 0, 1, 2
T0 = time.time()

In [ ]:
train_df, val_df = load_splits()
test_df = load_test()
y_tr, y_va = train_df.label.to_numpy(), val_df.label.to_numpy()
clip = lambda s, n=MAX_CHARS: s.astype(str).str.slice(0, n)
print(f"train={len(train_df):,}  val={len(val_df):,}  test={len(test_df):,}")

skf = StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED)
FOLDS = list(skf.split(np.zeros(len(y_tr)), train_df.domain + "|" + train_df.label))

## 0. Reproduce the 01 baseline (plus out-of-fold training predictions)

Same vectorizers and classifier as 01's winner. The **out-of-fold (OOF)** probabilities on the training split
(5 folds) are needed later: stacking and bias tuning must learn on predictions the base model did *not* see in
training, otherwise they would learn from over-confident scores. Runtime about 10 min. Peak RAM about 4 GB.

In [ ]:
VEC_PARAMS = {   # identical to 01_baseline (word_cased_punct + char_wb_3_5 = "word+char")
    "char_wb_3_5":      dict(analyzer="char_wb", ngram_range=(3, 5), min_df=5, max_features=300_000, lowercase=False),
    "word_cased_punct": dict(analyzer="word", ngram_range=(1, 2), min_df=3, max_features=200_000,
                             lowercase=False, token_pattern=TOKEN_PATTERN),
}
vecs, blocks = {}, {}
for name, kw in VEC_PARAMS.items():          # char first: its fit is the memory peak
    t0 = time.time()
    v = TfidfVectorizer(sublinear_tf=True, dtype=np.float32, **kw)
    blocks[name] = (v.fit_transform(clip(train_df.text)), v.transform(clip(val_df.text)), v.transform(clip(test_df.text)))
    vecs[name] = v
    print(f"{name}: {blocks[name][0].shape[1]:,} features [{time.time() - t0:.0f}s]")
    gc.collect()

Xw_tr, Xw_va, Xw_te = blocks["word_cased_punct"]          # word block kept separately for segment scoring
X_tr, X_va, X_te = (hstack_csr([blocks["word_cased_punct"][k], blocks["char_wb_3_5"][k]]) for k in range(3))
del blocks; gc.collect()
FEAT_NAMES = np.concatenate([vecs["word_cased_punct"].get_feature_names_out(),
                             vecs["char_wb_3_5"].get_feature_names_out()])

In [ ]:
def make_base():
    return LogisticRegression(C=4.0, class_weight="balanced", max_iter=2000)


def f1s(y, pred):
    f = f1_score(y, pred, labels=LABELS, average=None)
    return dict(macro=f.mean(), **{c: v for c, v in zip(LABELS, f)})


MODELS = {}   # name -> dict(macro, P_va, P_te, pred_va, pred_te)


def report(name, pred_va, pred_te=None, P_va=None, P_te=None, notes=""):
    log_result(EXP, name, y_va, pred_va, notes=notes)
    f = f1s(y_va, pred_va)
    print(f"{name:55s} macro={f['macro']:.4f}  human={f['human']:.4f}  fusion={f['fusion']:.4f}  ai={f['ai']:.4f}")
    MODELS[name] = dict(**f, P_va=P_va, P_te=P_te, pred_va=np.asarray(pred_va), pred_te=pred_te)


t0 = time.time()
base = make_base().fit(X_tr, y_tr)
assert list(base.classes_) == list(CLS)
P_va, P_te = base.predict_proba(X_va), base.predict_proba(X_te)
report("base LogReg C=4 | word+char (01 reproduced)", CLS[P_va.argmax(1)], CLS[P_te.argmax(1)], P_va, P_te,
       notes="reproduction of 01 best")
print(f"[{time.time() - t0:.0f}s]")

P_tr_oof = np.zeros((len(y_tr), 3), dtype=np.float32)
for k, (a, b) in enumerate(FOLDS):
    t0 = time.time()
    m = make_base().fit(X_tr[a], y_tr[a])
    P_tr_oof[b] = m.predict_proba(X_tr[b])
    print(f"OOF fold {k}: [{time.time() - t0:.0f}s]")
    del m; gc.collect()
print("OOF macro-F1 on train:", round(macro_f1(y_tr, CLS[P_tr_oof.argmax(1)]), 4))

## 1. Error inventory

In [ ]:
pred_va = CLS[P_va.argmax(1)]
ev = val_df[["domain", "lang"]].assign(y=y_va, pred=pred_va, n_chars=val_df.text.str.len())


def hf_table(g):
    nh, nf = max(1, (g.y == "human").sum()), max(1, (g.y == "fusion").sum())
    f = f1_score(g.y, g.pred, labels=["human", "fusion"], average=None, zero_division=0)
    return pd.Series({"n_human": (g.y == "human").sum(), "n_fusion": (g.y == "fusion").sum(),
                      "human→fusion": ((g.y == "human") & (g.pred == "fusion")).sum() / nh,
                      "fusion→human": ((g.y == "fusion") & (g.pred == "human")).sum() / nf,
                      "fusion→ai": ((g.y == "fusion") & (g.pred == "ai")).sum() / nf,
                      "human→ai": ((g.y == "human") & (g.pred == "ai")).sum() / nh,
                      "f1_human": f[0], "f1_fusion": f[1]})


for col in ("domain", "lang"):
    print(ev.groupby(col).apply(hf_table, include_groups=False).sort_values("f1_fusion").round(3).to_string(), "\n")

ev["len_bucket"] = pd.qcut(ev.n_chars, 5)
print(ev.groupby("len_bucket", observed=True).apply(hf_table, include_groups=False).round(3).to_string())

In [ ]:
q_va = P_va[:, iF] / (P_va[:, iF] + P_va[:, iH])       # P(fusion | human or fusion)
fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
for lab, c in (("human", "tab:blue"), ("fusion", "tab:orange")):
    ax[0].hist(q_va[y_va == lab], bins=50, alpha=.6, color=c, label=f"true {lab}")
ax[0].set(title="Baseline P(fusion | human∪fusion) on validation", xlabel="p", yscale="log"); ax[0].legend()
conf = np.maximum(P_va[:, iF], P_va[:, iH])[np.isin(y_va, ["human", "fusion"])]
ok = (pred_va == y_va)[np.isin(y_va, ["human", "fusion"])]
bins = np.linspace(.33, 1, 12); idx = np.digitize(conf, bins)
ax[1].plot([(conf[idx == i]).mean() for i in range(1, 12) if (idx == i).sum() > 20],
           [(ok[idx == i]).mean() for i in range(1, 12) if (idx == i).sum() > 20], "o-")
ax[1].plot([.4, 1], [.4, 1], "k--", lw=.8); ax[1].set(title="Reliability (human+fusion rows)", xlabel="confidence", ylabel="accuracy")
plt.tight_layout(); plt.show()

## 2. Read the misses

For each text: true and predicted label, probabilities, the **n-grams that pushed it towards fusion and towards
human** (TF-IDF value × (coef_fusion − coef_human)), then the first ~1200 characters. `⏎` marks a newline,
because layout is a signal too. One text per domain for each error direction, chosen by the largest wrong margin.

In [ ]:
W_FH = base.coef_[iF] - base.coef_[iH]


def why(i, k=7):
    row = X_va[i]
    c = row.data * W_FH[row.indices]
    o = np.argsort(c)
    fmt = lambda js: ", ".join(f"{FEAT_NAMES[row.indices[j]]!r}{c[j]:+.2f}" for j in js)
    return fmt(o[::-1][:k]), fmt(o[:k])


def show(rows, title, chars=1200):
    print(f"\n{'=' * 30} {title} ({len(rows)}) {'=' * 30}")
    for i in rows:
        r = val_df.iloc[i]
        to_f, to_h = why(i)
        print(f"\n--- id={r.id}  {r.domain} / {r.lang}  TRUE={r.label}  PRED={pred_va[i]}  "
              f"p(human/fusion/ai)={P_va[i, iH]:.2f}/{P_va[i, iF]:.2f}/{P_va[i, iA]:.2f}  chars={len(r.text)}")
        print(f"    → fusion: {to_f}\n    → human : {to_h}")
        print("    " + r.text[:chars].replace("\n", "⏎\n    ") + ("…" if len(r.text) > chars else ""))


def worst_per_domain(mask, margin, k=1):
    d = pd.DataFrame({"i": np.where(mask)[0], "m": margin[mask], "dom": val_df.domain.to_numpy()[mask]})
    return d.sort_values("m", ascending=False).groupby("dom").head(k).i.tolist()


margin = P_va[:, iF] - P_va[:, iH]
show(worst_per_domain((y_va == "human") & (pred_va == "fusion"), margin), "HUMAN predicted as FUSION")

In [ ]:
show(worst_per_domain((y_va == "fusion") & (pred_va == "human"), -margin), "FUSION predicted as HUMAN")

In [ ]:
rng = np.random.default_rng(SEED)
border = np.where(np.isin(y_va, ["human", "fusion"]) & (np.abs(margin) < 0.1))[0]
print(f"borderline human/fusion texts (|p_f - p_h| < 0.1): {len(border)}")
show(rng.choice(border, size=min(4, len(border)), replace=False), "BORDERLINE", chars=800)

### 2b. Misses that have a near-duplicate "twin" with the other label

`splits/near_dup_pairs.csv` (from 00) links validation texts to training texts that share ≥20% of word 5-grams.
A human↔fusion pair is very likely the original and its partial rewrite, so a word diff shows **exactly what the
LLM changed** in a text the model got wrong. `[-deleted-]` is human-only, `{+inserted+}` is from the rewrite.

In [ ]:
pairs = pd.read_csv(SPLITS_DIR / "near_dup_pairs.csv")
TOK = re.compile(TOKEN_PATTERN)
miss_ids = set(val_df.id[np.isin(y_va, ["human", "fusion"]) & (pred_va != y_va)])
tw = pairs[pairs.val_id.isin(miss_ids) & (pairs.train_label != pairs.val_label)
           & pairs.train_label.isin(["human", "fusion"])].sort_values("jaccard", ascending=False)
tw = tw.drop_duplicates("val_id")
print(f"misclassified human/fusion val texts with an opposite-label twin: {len(tw)} of {len(miss_ids)}")


def render_diff(human: str, fusion: str, width=1100, max_tok=1500):
    a, b = TOK.findall(human)[:max_tok], TOK.findall(fusion)[:max_tok]
    sm = difflib.SequenceMatcher(None, a, b, autojunk=False)
    out = []
    for t, i1, i2, j1, j2 in sm.get_opcodes():
        if t == "equal":
            seg = a[i1:i2]
            out.append(" ".join(seg) if len(seg) < 40 else " ".join(seg[:12]) + " … " + " ".join(seg[-12:]))
        else:
            if i2 > i1: out.append("[-" + " ".join(a[i1:i2]) + "-]")
            if j2 > j1: out.append("{+" + " ".join(b[j1:j2]) + "+}")
    return f"(kept {sm.ratio():.0%} of tokens) " + " ".join(out)[:width]


if len(tw):
    twin_txt = load_rows(tw.train_id.head(6)).set_index("id").text
    vtxt = val_df.set_index("id").text
    for _, r in tw.head(6).iterrows():
        h, f = (twin_txt[r.train_id], vtxt[r.val_id]) if r.train_label == "human" else (vtxt[r.val_id], twin_txt[r.train_id])
        print(f"\n--- val {r.val_id} TRUE={r.val_label}  twin {r.train_id}={r.train_label}  {r.domain}  J={r.jaccard:.2f}")
        print("   ", render_diff(h, f))

## 3. Outside research: what to test

Ideas from the literature, each turned into a measurable feature in §4. None is assumed to hold on this data.

* **Machine-polished human text is its own category.** *LLM-DetectAIve* (Abassy et al., EMNLP 2024 demo,
  [arXiv 2408.04284](https://arxiv.org/abs/2408.04284)) separates *human-written, machine-polished* text from
  human and machine text. This is our *fusion*. The polishing step fixes grammar and spelling and smooths the style,
  so **lost human noise** (typos, informal punctuation) should be a signal. → misspelling and typography features.
* **Hybrid texts are a boundary problem.** Zeng et al., *Towards Automatic Boundary Detection for Human-AI
  Collaborative Hybrid Essay* (AAAI 2024, [arXiv 2307.12267](https://arxiv.org/abs/2307.12267)), SemEval-2024
  Task 8 subtask C ([DeepPavlov system](https://arxiv.org/html/2405.10629v1)) and RoFT boundary detection
  ([arXiv 2311.08349](https://arxiv.org/abs/2311.08349)) all treat mixed text as *segments* with different
  authors. → **segment-level** scores: variance, maximum, and share of AI-looking chunks inside one text.
* **LLM style is more uniform.** Zanotto & Aroyehun (EMNLP 2025, [arXiv 2507.13614](https://arxiv.org/html/2507.13614v3))
  report more standardised, homogeneous LLM text. Stylometry on short samples
  ([arXiv 2507.00838](https://arxiv.org/abs/2507.00838)) reports that LLMs vary word length less. →
  **burstiness**: sentence-length variance, word-length spread, type-token ratio.
* **Excess vocabulary.** Juzek & Ward, *Why does ChatGPT "delve" so much?* (COLING 2025,
  [arXiv 2412.11385](https://arxiv.org/abs/2412.11385)), and Kobak et al.'s "excess words" in abstracts. →
  rate of LLM **stock words** (*delve, intricate, underscore, pivotal, tapestry, moreover…*, plus German
  equivalents).

## 4. Hand-crafted "human-ness" features

### 4a. Misspellings

`pyspellchecker` (English and German frequency dictionaries). Only **lowercase-initial** alphabetic tokens of
length ≥ 3 are checked, so names and German nouns do not count as typos. A second, data-driven dictionary is
also built: words that occur in ≥ 3 *AI* training documents. A word outside it is "something an LLM never
writes". The spell check runs once over the unique vocabulary, not per text.

In [ ]:
WORD = re.compile(r"[^\W\d_]+")
cand = lambda t: [w for w in WORD.findall(t) if len(w) >= 3 and w[0].islower()]

SPELL = {"en-EN": SpellChecker(language="en"), "de-DE": SpellChecker(language="de")}
MISSPELLED = {}
t0 = time.time()
for lang, sc in SPELL.items():
    vocab = set()
    for df in (train_df, val_df, test_df):
        for t in df.text[df.lang == lang]:
            vocab.update(w.lower() for w in cand(t[:FEAT_CHARS]))
    MISSPELLED[lang] = sc.unknown(vocab)
    print(f"{lang}: vocab={len(vocab):,}  unknown to dictionary={len(MISSPELLED[lang]):,}")
print(f"[{time.time() - t0:.0f}s]")

ai_df = Counter()
for t in train_df.text[train_df.label == "ai"]:
    ai_df.update({w.lower() for w in cand(t[:FEAT_CHARS])})
AI_VOCAB = {w for w, c in ai_df.items() if c >= 3}
print(f"AI vocabulary (df>=3 in ai training texts): {len(AI_VOCAB):,} words")

In [ ]:
def typo_list(t, lang):
    ms = MISSPELLED[lang]
    return [w.lower() for w in cand(t) if w.lower() in ms]


# Which misspelled tokens are typical of each class? Document frequency on the TRAINING split
tdf = {c: Counter() for c in LABELS}
for t, lang, lab in zip(train_df.text, train_df.lang, train_df.label):
    tdf[lab].update(set(typo_list(t[:FEAT_CHARS], lang)))
n_lab = train_df.label.value_counts()
tt = pd.DataFrame({c: pd.Series(tdf[c]) for c in LABELS}).fillna(0).astype(int)
for c in LABELS:
    tt[f"{c}_per1k_docs"] = (tt[c] / n_lab[c] * 1000).round(2)
tt["total"] = tt[LABELS].sum(1)
print("Most common 'misspellings' overall (many are dictionary gaps, not typos):")
print(tt.sort_values("total", ascending=False).head(25).to_string())

In [ ]:
# "Human-only" typos: misspelled in >= 3 human training docs and never in an ai training doc
HUMAN_ONLY = set(tt.index[(tt.human >= 3) & (tt.ai == 0)])
print(f"human-only misspellings (train): {len(HUMAN_ONLY):,}  e.g.",
      ", ".join(tt.loc[list(HUMAN_ONLY)].sort_values("human", ascending=False).index[:40]))

# Test on VALIDATION: how often does a text contain at least one such token, per class?
has = np.array([bool(set(typo_list(t[:FEAT_CHARS], l)) & HUMAN_ONLY) for t, l in zip(val_df.text, val_df.lang)])
print("\nShare of validation texts containing >=1 human-only typo:")
print(pd.crosstab(val_df.domain, val_df.label, values=has, aggfunc="mean").round(3).to_string())
print("\nP(label | contains a human-only typo) on validation:")
print(pd.Series(y_va[has]).value_counts(normalize=True).round(3).to_string())

### 4b. Typography, burstiness, stock words, and segment inconsistency

All counts are **per 100 words**. The segment features split each text (first 8k chars) into chunks of about 80
words at sentence boundaries. Each chunk is scored by an **ai-vs-human** word TF-IDF LogReg. Training-split scores
are out-of-fold, so the model never scores its own training text. A fusion text should then contain *some*
AI-looking chunks and *some* human-looking ones.

In [ ]:
STOCK_EN = ("delve delves delving intricate underscore underscores pivotal tapestry realm meticulous meticulously "
            "showcase showcasing notably moreover furthermore additionally crucial comprehensive foster fostering "
            "navigate navigating landscape testament multifaceted vibrant seamless seamlessly enhance enhancing "
            "leverage robust nuanced commendable embark profound insights ultimately overall paramount "
            "noteworthy invaluable endeavor endeavors resonate resonates bolster harness").split()
STOCK_DE = ("zudem darüber hinaus entscheidend maßgeblich insbesondere letztendlich vielfältig vielfältige "
            "nachhaltig umfassend umfassende wesentlich wesentliche hervorzuheben bemerkenswert zentral "
            "zentrale essenziell essentiell gleichermaßen unterstreicht verdeutlicht").split()
STOCK = re.compile(r"\b(" + "|".join(sorted(set(STOCK_EN + STOCK_DE), key=len, reverse=True)) + r")\b", re.I)
PHRASES = re.compile(r"in conclusion|it is important to note|it is worth noting|in summary|plays a crucial role|"
                     r"a testament to|abschließend|es ist wichtig|zusammenfassend", re.I)
SENT = re.compile(r"(?<=[.!?…])\s+|\n+")

PATTERNS = {   # name -> regex, counted per 100 words
    "lower_i": r"(?<![\w'’])i(?![\w'’])", "dbl_punct": r"[!?]{2,}", "ellipsis": r"\.\.\.|…",
    "space_before_punct": r"\w\s+[,.;:!?](?!\w)", "no_space_after_punct": r"[a-zäöü][,;:!?][A-Za-zÄÖÜ]",
    "allcaps": r"\b[A-ZÄÖÜ]{3,}\b", "emoticon": r"[:;]-?[)(DPp]\B", "contraction": r"\w['’](?:s|t|re|ve|ll|d|m)\b",
    "curly_quote": r"[’“”‘]", "straight_quote": r"['\"]", "em_dash": r"—", "en_dash": r"–", "spaced_hyphen": r" - ",
    "newline": r"\n", "dbl_newline": r"\n\s*\n", "dbl_space": r"(?<=\S)  +(?=\S)", "lower_sent_start": r"[.!?]\s+[a-zäöü]",
    "digit": r"\d", "paren": r"\(", "exclam": r"!", "question": r"\?", "semicolon": r";", "colon": r":",
    "comma": r",", "stock_word": STOCK.pattern, "stock_phrase": PHRASES.pattern,
}
PATTERNS_RE = {k: re.compile(v, re.I if k.startswith("stock") else 0) for k, v in PATTERNS.items()}


def text_features(t: str, lang: str) -> dict:
    t = t[:FEAT_CHARS]
    words = t.split(); nw = max(1, len(words))
    f = {k: len(r.findall(t)) / nw * 100 for k, r in PATTERNS_RE.items()}
    sl = [len(s.split()) for s in SENT.split(t) if s.strip()] or [0]
    wl = [len(w) for w in words] or [0]
    c = cand(t); low = [w.lower() for w in c]; nc = max(1, len(c))
    typos = [w for w in low if w in MISSPELLED[lang]]
    head = [w.lower() for w in words[:300]]
    f.update(
        n_words=np.log1p(len(words)), sent_len_mean=np.mean(sl), sent_len_std=np.std(sl),
        sent_len_cv=np.std(sl) / (np.mean(sl) + 1e-6), sent_len_max=max(sl),
        word_len_mean=np.mean(wl), word_len_std=np.std(wl), ttr_300=len(set(head)) / max(1, len(head)),
        typo_rate=len(typos) / nc * 100, typo_count=len(typos),
        human_only_typo=sum(w in HUMAN_ONLY for w in typos),
        oov_ai_rate=sum(w not in AI_VOCAB for w in low) / nc * 100,
        curly_share=f["curly_quote"] / (f["curly_quote"] + f["straight_quote"] + 1e-6),
    )
    return f


t0 = time.time()
D_tr, D_va, D_te = (pd.DataFrame([text_features(t, l) for t, l in zip(df.text, df.lang)]) for df in (train_df, val_df, test_df))
print(f"{D_tr.shape[1]} text features [{time.time() - t0:.0f}s]")

In [ ]:
CHUNK_WORDS = 80


def chunks(t: str) -> list:
    out, cur, n = [], [], 0
    for s in SENT.split(t[:FEAT_CHARS]):
        if not s.strip():
            continue
        cur.append(s); n += len(s.split())
        if n >= CHUNK_WORDS:
            out.append(" ".join(cur)); cur, n = [], 0
    if cur:
        if out and n < CHUNK_WORDS / 2: out[-1] += " " + " ".join(cur)
        else: out.append(" ".join(cur))
    return out or [t[:FEAT_CHARS]]


def chunk_table(df):
    rows = [(i, c) for i, t in enumerate(df.text) for c in chunks(t)]
    ct = pd.DataFrame(rows, columns=["doc", "text"])
    langs = df.lang.to_numpy()[ct.doc]
    ct["typo"] = [len(typo_list(c, l)) / max(1, len(cand(c))) * 100 for c, l in zip(ct.text, langs)]
    return ct


def seg_model():
    return LogisticRegression(C=4.0, class_weight="balanced", max_iter=2000)


t0 = time.time()
CT = {"tr": chunk_table(train_df), "va": chunk_table(val_df), "te": chunk_table(test_df)}
XC = {k: vecs["word_cased_punct"].transform(v.text) for k, v in CT.items()}
print("chunks:", {k: len(v) for k, v in CT.items()}, f"[{time.time() - t0:.0f}s]")

ah = np.isin(y_tr, ["ai", "human"])
CT["tr"]["p_ai"] = np.nan
for k, (a, b) in enumerate(FOLDS):           # OOF: chunks of fold b scored by a model trained on fold a
    a = a[ah[a]]
    m = seg_model().fit(Xw_tr[a], y_tr[a] == "ai")
    sel = np.isin(CT["tr"].doc.to_numpy(), b)
    CT["tr"].loc[sel, "p_ai"] = m.predict_proba(XC["tr"][sel])[:, 1]
m = seg_model().fit(Xw_tr[ah], y_tr[ah] == "ai")
for k in ("va", "te"):
    CT[k]["p_ai"] = m.predict_proba(XC[k])[:, 1]
print(f"segment scoring done [{time.time() - t0:.0f}s]")


def seg_features(ct, n):
    g = ct.groupby("doc")
    s = pd.DataFrame({
        "seg_n": g.size(), "seg_pai_mean": g.p_ai.mean(), "seg_pai_std": g.p_ai.std().fillna(0),
        "seg_pai_max": g.p_ai.max(), "seg_pai_min": g.p_ai.min(),
        "seg_share_ai": g.p_ai.apply(lambda x: (x > .5).mean()),
        "seg_share_human": g.p_ai.apply(lambda x: (x < .2).mean()),
        "seg_typo_std": g.typo.std().fillna(0), "seg_typo_max": g.typo.max(),
        "seg_share_typo_free": g.typo.apply(lambda x: (x == 0).mean()),
    })
    s["seg_pai_range"] = s.seg_pai_max - s.seg_pai_min
    return s.reindex(range(n)).reset_index(drop=True)


S_tr, S_va, S_te = seg_features(CT["tr"], len(train_df)), seg_features(CT["va"], len(val_df)), seg_features(CT["te"], len(test_df))
F_tr, F_va, F_te = (pd.concat([D, S], axis=1) for D, S in ((D_tr, S_tr), (D_va, S_va), (D_te, S_te)))
del XC; gc.collect()

### 4c. Which features separate human from fusion? (per-domain AUC, training split)

AUC of the raw feature for human (0) vs fusion (1): **< 0.5 means higher in human, > 0.5 means higher in fusion**.
Domains need ≥ 100 texts of each class. The first row is the baseline's own OOF `P(fusion | h∪f)` as a reference
for how strong a single feature would have to be.

In [ ]:
hf = np.isin(y_tr, ["human", "fusion"])
ref = pd.Series(P_tr_oof[:, iF] / (P_tr_oof[:, iF] + P_tr_oof[:, iH]), name="BASELINE p_fusion (OOF)")
G = pd.concat([ref, F_tr], axis=1)[hf].assign(_y=(y_tr[hf] == "fusion").astype(int), _d=train_df.domain.to_numpy()[hf])
doms = [d for d, g in G.groupby("_d") if min((g._y == 1).sum(), (g._y == 0).sum()) >= 100]
auc = pd.DataFrame({d: {c: roc_auc_score(g._y, g[c].fillna(0)) for c in G.columns[:-2]}
                    for d, g in G.groupby("_d") if d in doms})
auc["all"] = [roc_auc_score(G._y, G[c].fillna(0)) for c in auc.index]
auc = auc.loc[(auc["all"] - .5).abs().sort_values(ascending=False).index]
top = auc.head(30)
fig, ax = plt.subplots(figsize=(10, .32 * len(top) + 1.5))
im = ax.imshow(top.values, cmap="RdBu_r", vmin=.2, vmax=.8, aspect="auto")
ax.set_xticks(range(top.shape[1]), top.columns, rotation=30, ha="right"); ax.set_yticks(range(len(top)), top.index)
for i in range(top.shape[0]):
    for j in range(top.shape[1]):
        ax.text(j, i, f"{top.values[i, j]:.2f}", ha="center", va="center", fontsize=7)
fig.colorbar(im, ax=ax, label="AUC  (red = higher in fusion, blue = higher in human)")
ax.set_title("Human vs fusion separability per domain (training split)"); plt.tight_layout(); plt.show()

In [ ]:
# Do the features add information *beyond* the baseline? Partial check: AUC of each feature among human/fusion
# texts on which the baseline is unsure (OOF p_fusion in [0.25, 0.75]).
unsure = G[(G.iloc[:, 0] > .25) & (G.iloc[:, 0] < .75)]
print(f"unsure human/fusion training texts: {len(unsure):,}  (fusion share {unsure._y.mean():.2f})")
ua = pd.Series({c: roc_auc_score(unsure._y, unsure[c].fillna(0)) for c in G.columns[1:-2]})
print(ua.sub(.5).abs().sort_values(ascending=False).head(15).add(.5).round(3).to_string())
print("\nClass medians of the 12 strongest features (training split):")
F12 = [c for c in auc.index if c in F_tr.columns][:12]
print(pd.concat([F_tr, ref], axis=1).assign(label=y_tr).groupby("label")[list(F12)].median().loc[LABELS].T.round(3).to_string())

## 5. Models that use the findings

| id | Model | Idea tested |
|---|---|---|
| a | TF-IDF + standardised hand-crafted/segment features + domain one-hot → LogReg | do the features help when added directly? |
| c | + **"text shape"** char n-grams (letters → `a`/`A`, digits → `0`, punctuation and whitespace kept) | typo- and spacing-style artifacts that `char_wb` partly drops |
| e | **Stacker**: gradient boosting on OOF base probabilities + features + domain | non-linear interactions such as "typo-free in blogs" |
| b | **Two-stage**: base model, then a dedicated human-vs-fusion booster for rows predicted human/fusion | specialise on the hard pair |
| d | **Bias tuning**: shift human vs fusion probability, global or per domain, tuned on OOF training predictions | fix calibration and prior shifts per domain |

In [ ]:
DOMS = sorted(train_df.domain.unique())
dom_oh = lambda df: sp.csr_matrix(pd.get_dummies(pd.Categorical(df.domain, categories=DOMS)).to_numpy(np.float32))
scaler = StandardScaler().fit(F_tr.fillna(0))
dense = lambda F: sp.csr_matrix(np.clip(scaler.transform(F.fillna(0)), -5, 5).astype(np.float32))

t0 = time.time()
A_tr, A_va, A_te = (hstack_csr([X, dense(F), dom_oh(df)]) for X, F, df in
                    ((X_tr, F_tr, train_df), (X_va, F_va, val_df), (X_te, F_te, test_df)))
ma = make_base().fit(A_tr, y_tr)
Pa_va, Pa_te = ma.predict_proba(A_va), ma.predict_proba(A_te)
report("a) word+char + features + domain | LogReg C=4", CLS[Pa_va.argmax(1)], CLS[Pa_te.argmax(1)], Pa_va, Pa_te,
       notes="hand-crafted + segment features as extra columns")
print(f"[{time.time() - t0:.0f}s]")
del A_tr, A_va, A_te, ma; gc.collect()

In [ ]:
def shape(t: str) -> str:
    t = re.sub(r"[A-ZÄÖÜ]", "A", t[:MAX_CHARS])
    t = re.sub(r"[^\W\d_A]", "a", t)
    t = re.sub(r"\d", "0", t)
    return re.sub(r"([aA0])\1{2,}", r"\1\1\1", t)


print("example:", repr(shape(val_df.text.iloc[0])[:200]))
t0 = time.time()
shape_vec = TfidfVectorizer(analyzer="char", ngram_range=(2, 6), min_df=5, max_features=100_000,
                            lowercase=False, sublinear_tf=True, dtype=np.float32)
Z_tr = shape_vec.fit_transform(train_df.text.map(shape))
Z_va, Z_te = shape_vec.transform(val_df.text.map(shape)), shape_vec.transform(test_df.text.map(shape))
print(f"shape features: {Z_tr.shape[1]:,}  nnz/row={Z_tr.nnz / Z_tr.shape[0]:.0f}  [{time.time() - t0:.0f}s]")

# alone (how much signal the shape carries), with OOF for the stacker
ms = make_base().fit(Z_tr, y_tr)
Ps_va, Ps_te = ms.predict_proba(Z_va), ms.predict_proba(Z_te)
report("c0) text-shape char 2-6 only | LogReg C=4", CLS[Ps_va.argmax(1)], notes="shape n-grams alone")
Ps_tr_oof = np.zeros_like(P_tr_oof)
for a, b in FOLDS:
    Ps_tr_oof[b] = make_base().fit(Z_tr[a], y_tr[a]).predict_proba(Z_tr[b])

t0 = time.time()
C_tr, C_va, C_te = (hstack_csr([X, Z, dense(F), dom_oh(df)]) for X, Z, F, df in
                    ((X_tr, Z_tr, F_tr, train_df), (X_va, Z_va, F_va, val_df), (X_te, Z_te, F_te, test_df)))
mc = make_base().fit(C_tr, y_tr)
Pc_va, Pc_te = mc.predict_proba(C_va), mc.predict_proba(C_te)
report("c) word+char + shape + features + domain | LogReg C=4", CLS[Pc_va.argmax(1)], CLS[Pc_te.argmax(1)], Pc_va, Pc_te,
       notes="adds text-shape n-grams")
print(f"[{time.time() - t0:.0f}s]")
del C_tr, C_va, C_te, mc; gc.collect()

In [ ]:
def meta(P, Ps, F, df):
    lo = lambda p: np.log(p + 1e-6)
    m = pd.DataFrame({"p_ai": P[:, iA], "p_fusion": P[:, iF], "p_human": P[:, iH],
                      "lo_f_h": lo(P[:, iF]) - lo(P[:, iH]), "lo_ai_f": lo(P[:, iA]) - lo(P[:, iF]),
                      "shape_p_ai": Ps[:, iA], "shape_lo_f_h": lo(Ps[:, iF]) - lo(Ps[:, iH])})
    m = pd.concat([m, F.reset_index(drop=True)], axis=1)
    m["domain"] = pd.Categorical(df.domain.to_numpy(), categories=DOMS).codes
    m["lang"] = (df.lang.to_numpy() == "de-DE").astype(int)
    return m


M_tr, M_va, M_te = meta(P_tr_oof, Ps_tr_oof, F_tr, train_df), meta(P_va, Ps_va, F_va, val_df), meta(P_te, Ps_te, F_te, test_df)
PROBS_ONLY = ["p_ai", "p_fusion", "p_human", "lo_f_h", "lo_ai_f", "domain"]


def hgb(cols):
    return HistGradientBoostingClassifier(max_iter=600, learning_rate=0.05, max_leaf_nodes=31, min_samples_leaf=40,
                                          l2_regularization=1.0, early_stopping=True, validation_fraction=0.1,
                                          n_iter_no_change=40, random_state=SEED,
                                          categorical_features=[cols.index("domain")])


for tag, cols in (("e0) stacker: base probs + domain only", PROBS_ONLY),
                  ("e) stacker: base probs + shape + features + domain", list(M_tr.columns))):
    t0 = time.time()
    h = hgb(cols).fit(M_tr[cols], y_tr)
    Pe_va, Pe_te = h.predict_proba(M_va[cols]), h.predict_proba(M_te[cols])
    report(tag, CLS[Pe_va.argmax(1)], CLS[Pe_te.argmax(1)], Pe_va, Pe_te, notes="HistGradientBoosting on OOF meta-features")
    print(f"   iterations={h.n_iter_}  [{time.time() - t0:.0f}s]")
STACK_COLS = list(M_tr.columns)

In [ ]:
# b) two-stage: keep the base model's "ai" decisions; a dedicated human-vs-fusion booster decides the rest
hf_tr = np.isin(y_tr, ["human", "fusion"])
two = hgb(STACK_COLS).fit(M_tr[hf_tr], y_tr[hf_tr])


def two_stage(P, M):
    pred = CLS[P.argmax(1)].astype(object)
    sel = pred != "ai"
    pred[sel] = two.predict(M[sel])
    return pred


report("b) two-stage: base → human-vs-fusion booster", two_stage(P_va, M_va), two_stage(P_te, M_te),
       notes="stage 2 trained on human+fusion OOF meta-features")

In [ ]:
# d) bias tuning on OOF training predictions: multiply P(human) by w (global or per domain), argmax
W_GRID = np.exp(np.linspace(-1.5, 1.5, 61))


def apply_w(P, w):
    Q = P.copy(); Q[:, iH] *= w; return CLS[Q.argmax(1)]


def tune(P_oof, doms_tr):
    g = max(W_GRID, key=lambda w: macro_f1(y_tr, apply_w(P_oof, w)))
    per = {}
    for d in DOMS:
        m = doms_tr == d
        if min((y_tr[m] == "human").sum(), (y_tr[m] == "fusion").sum()) < 200:
            per[d] = g; continue
        per[d] = max(W_GRID, key=lambda w: macro_f1(y_tr[m], apply_w(P_oof[m], w)))
    return g, per


def apply_per(P, df, per):
    return apply_w(P, df.domain.map(per).to_numpy())


dom_tr = train_df.domain.to_numpy()
g, per = tune(P_tr_oof, dom_tr)
print(f"base: global w={g:.2f}  per-domain:", {d: round(w, 2) for d, w in per.items()})
report("d1) base + global human/fusion bias (OOF-tuned)", apply_w(P_va, g), apply_w(P_te, g), notes=f"w={g:.2f}")
report("d2) base + per-domain bias (OOF-tuned)", apply_per(P_va, val_df, per), apply_per(P_te, test_df, per), notes="per-domain w")

# the same on top of the stacker, which needs its own OOF predictions on the training split
Pe_tr_oof = np.zeros_like(P_tr_oof)
for a, b in FOLDS:
    Pe_tr_oof[b] = hgb(STACK_COLS).fit(M_tr.iloc[a], y_tr[a]).predict_proba(M_tr.iloc[b])
h = hgb(STACK_COLS).fit(M_tr, y_tr)
Pe_va, Pe_te = h.predict_proba(M_va), h.predict_proba(M_te)
g2, per2 = tune(Pe_tr_oof, dom_tr)
print(f"stacker: global w={g2:.2f}  per-domain:", {d: round(w, 2) for d, w in per2.items()})
report("d3) stacker + per-domain bias (OOF-tuned)", apply_per(Pe_va, val_df, per2), apply_per(Pe_te, test_df, per2),
       Pe_va, Pe_te, notes="per-domain w on stacker")

In [ ]:
# Which features does the stacker use? Permutation importance on validation human+fusion rows (macro-F1 drop)
from sklearn.inspection import permutation_importance

sel = np.isin(y_va, ["human", "fusion"])
pi = permutation_importance(h, M_va[sel], y_va[sel], n_repeats=3, random_state=SEED, n_jobs=1,
                            scoring=lambda est, X, y: f1_score(y, est.predict(X), labels=["human", "fusion"], average="macro"))
imp = pd.Series(pi.importances_mean, index=STACK_COLS).sort_values(ascending=False)
print("Top 20 stacker features (drop in human/fusion macro-F1 when shuffled):")
print(imp.head(20).round(4).to_string())

## 6. Summary

In [ ]:
res = pd.DataFrame({k: {c: v[c] for c in ("macro", "human", "fusion", "ai")} for k, v in MODELS.items()}).T
ref01 = pd.read_csv(OUTPUT_DIR / "results.csv").query("experiment == '01_baseline'").macro_f1.max()
res["Δ macro vs 01 best"] = res.macro - ref01
print(f"01 best macro-F1 in results.csv: {ref01:.4f}\n")
print(res.sort_values("macro", ascending=False).round(4).to_string())

fig, ax = plt.subplots(figsize=(9, .35 * len(res) + 1))
r = res.sort_values("macro")
y = np.arange(len(r))
ax.scatter(r.human, y, label="f1_human", marker="o"); ax.scatter(r.fusion, y, label="f1_fusion", marker="s")
ax.scatter(r.macro, y, label="macro-F1", marker="|", s=200, c="k")
ax.set_yticks(y, r.index); ax.legend(loc="lower right"); ax.set_title("Validation F1 by experiment")
plt.tight_layout(); plt.show()

In [ ]:
best_name = res.macro.idxmax()
best = MODELS[best_name]
print(f"best: {best_name}  macro={best['macro']:.4f}")
if best["P_va"] is not None:
    save_val_scores("02_hf_best", val_df.id, best["P_va"], CLS)
if best["macro"] > ref01 and best["pred_te"] is not None:
    tag = re.sub(r"[^\w]+", "_", best_name.split(")")[0])
    save_submission(test_df.id, best["pred_te"], f"02_hf_{tag}")
else:
    print("no submission written (does not beat 01 or has no test predictions)")
print(f"total runtime {(time.time() - T0) / 60:.1f} min")

## Conclusions

*Filled in from the outputs above. Re-check after any rerun.*